# 🚀 Evaluate the distilled shopping agent on SageMaker AI: ShoppingBench ASR and IFEval

Runs three models as agents on the same held-out ShoppingBench problems:

| Model | Serving Platform |
|---|---|
| **Base** - Qwen3.5-4B Base Model before distllation | SageMaker AI real-time endpoint |
| **Student** - The distilled model | SageMaker AI real-time endpoint |
| **Reference Frontier Model** (optional) - Claude Sonnet 4.6 | Amazon Bedrock |

Reported per model: **ASR** (Absolute Success Rate of solving user shopping queries) and, optionally, **IFEval**
(General instruction following). 


## Shopping Agent Loop Per Trajectory


At every step the model receives a system prompt with the tool descriptions and the dialogue history. The model's response is in following format:


```<think> reasoning ... </think><tool_call>[{"name": "find_product", "parameters": {"q": "baggy high waist denim jeans", "page": 1}}]</tool_call>```


```<think>...</think>``` is included when thinking for the model is turned on. 


The harness parses the `<tool_call>` block, runs the tools, appends their output to the tool call history in the prompt. The agent acts on this new prompt with the tool call history by predicting it's tool calling. A
trajectory ends when the agent calls `terminate`, or after maximum 30 steps, or when the reply cannot be parsed (no `<tool_call>` or
`<response>` tag). The predicted product in the terminal state is then compared to the target product.

Along with the query, it stores the constraints that the product satisfies. An example:

```json
{"query": "Show me women's jeans in size eu:30 with a baggy fit, high waist, made of denim, plain design, and priced above 114 PHP.",
 "reward": {"product_id": "5113553158",
            "title": ["American Retro Baggy Jeans For Women High Waist Straight Denim ..."],
            "sku_options": [{"size": "eu:30"}],
            "attributes": [{"jeans_fit_type": ["baggy fit"]}, {"waist_type": ["high"]}, {"clothing_material": ["denim"]}, {"fa_pattern": ["plain"]}],
            "price": [{"greater than": [114, null]}]}}
```


### Tools available to agent
All run against a local shopping catalog server:

| Tool | Parameters | Returns |
|---|---|---|
| find_product | q: search text, page: 1–5<br>price: "low-high", e.g. "100-500"<br>service: official, freeShipping, COD, flashsale<br>sort: priceasc, pricedesc, order = most sold<br>shop_id: one shop only | Up to 10 products: id, shop id, title, price, services, sold count |
| view_product_information | product_ids: String of Product IDs e.g. "4410867766,4422385859" | Full details: attributes and SKU options (size, color, ...) |
| recommend_product | product_ids: String of Product IDs | This is the agent's response to the user query |
| terminate | status: e.g. success | Agent returns terminate to end the trajectory |


## Agent loop Evaluation and Inference 

The recommended product does **not** have to be the gold one. It is checked against every stored constraint:

| Check | Passes when |
|---|---|
| title | Semantic similarity between the recommended and gold titles is ≥ 0.5 (cosine, `Qwen3-Embedding-0.6B` embeddings), i.e. the same kind of product |
| price | Each bound holds: `less than`, `greater than`, or `between` |
| service | Every required service (e.g. `freeShipping`, `COD`) is offered |
| SKU options + attributes | Every required (key, value) pair, e.g. `size = eu:30`, `waist_type = high`, is found in the product's attributes or in one of its SKU variants (the best-matching variant counts) |

Recommending the exact gold product passes directly. Whether a whole trajectory is solved then follows the
"Benchmark Criteria" column of the intents table below 

## ShoppingBench Categories and Metrics

[ShoppingBench](https://github.com/yjwjy/ShoppingBench) ([paper](https://arxiv.org/abs/2508.04266)) gives an agent a
shopper's request and tools over "a large-scale shopping sandbox incorporating over 2.5 million real-world
products". The agent must search, inspect products and recommend the one(s) that meet every requirement. 

We measure three user **intents** of increasing difficulty `product`, `shop`, `voucher`. 
`EVAL_TASKS` in the configuration is a list that used to picks the intents to measure ASR on:

| Intent Type | Shopper Requirement | Benchmark Criteria  | Example |
|---|---|---|---|
| product | Find one product meeting a set of constraints | The recommended product meets all requirements | "Women's jeans in size eu:30, baggy fit, high waist, denim, plain, above 114 PHP." |
| shop  | Find Several products from one shop | Every recommended product meets all requirements, and all come from one shop | "Find shops offering gold jewelry that include: a necklace with zircon gemstones, a gold ring ..." |
| voucher | Find several products where total bill after a voucher is applied, fits the budget | Every requested product meets all requirements, and total bill after voucher is applied is within budget | "...a phone strap, a fuel pump filter, and an HP black ink bottle. My budget is only 398, but I have a voucher ..." |

**Metrics reported**

| Metric | Meaning |
|---|---|
| ASR | Absolute Success Rate: share of problems successfully completed |
| exact_id | Share where the exact gold product was recommended (stricter; many valid products exist) |
| recommend | Share of trajectories that recommended at least one real product (the base model often stops on an unparsable reply) |

These rules are ShoppingBench's own rule-based scorer (`src/agent/rewards/orm.py`, applied by
`src/agent/run_evaluate.py`); `scripts/score_rollouts.py` calls those same functions.

**What this notebook and scripts does** :

1. Set up the benchmark harness, product catalog and local search server — `setup_shopbench.sh`, `shopbench_harness.py`
2. Pick the held-out test problems (queries the training data never saw) — `make_clean_split.py`
3. Deploy the base and student endpoints — `sm_endpoints.py`
4. Point the harness at them through a local proxy that signs each request (or calls Bedrock) — `sm_openai_relay.py`
5. Run every model through the agent loop and score the results — `shopbench_harness.py`, `score_rollouts.py`
6. Optionally measure instruction following — `run_ifeval.py`
7. Delete the endpoints

## 0. Environment (once, then switch kernel)

Installs the notebook's packages into a venv registered as the **seqkd-eval** kernel (the harness and lm-eval get their
own venvs later). The harness and scoring run on this machine (CPU; e.g. a Studio `ml.m5.4xlarge` space or a laptop);
only the models run on SageMaker. Run the next cell, switch the kernel to **seqkd-eval**, and continue from 0b.

In [ ]:
import subprocess, sys
from pathlib import Path
WORK = Path.home() / "shopbench-eval"; WORK.mkdir(exist_ok=True)
EVAL_ENV = WORK / "evalenv"
if not (EVAL_ENV / "bin" / "python").exists():
    subprocess.run([sys.executable, "-m", "venv", str(EVAL_ENV)], check=True)
subprocess.run([str(EVAL_ENV / "bin/pip"), "install", "-q", "-U", "pip", "sagemaker>=3", "openai", "huggingface_hub",
                "transformers", "pandas", "matplotlib", "httpx", "requests", "boto3", "ipykernel"], check=True)
subprocess.run([str(EVAL_ENV / "bin/python"), "-m", "ipykernel", "install", "--user", "--name", "seqkd-eval",
                "--display-name", "seqkd-eval"], check=True)
print("Kernel 'seqkd-eval' registered. Switch to it (Kernel > Change Kernel), then run the cells below.")

## 0b. Configuration

The execution role needs: create/invoke SageMaker models and endpoints, `sagemaker:CallWithBearerToken` (resource
`"*"`), `bedrock:InvokeModel` for the reference model, and read access to the Model Package's S3 bucket (without it the
endpoint fails with only "Request to service failed").

In [ ]:
import os, json, sys, time, subprocess, logging
from pathlib import Path
assert "evalenv" in sys.executable, f"switch the kernel to 'seqkd-eval' first (current: {sys.executable})"
import boto3

REGION = boto3.Session().region_name or "us-west-2"
from sagemaker.core.helper.session_helper import get_execution_role
try:
    ROLE = get_execution_role()                 # inside SageMaker Studio / a notebook instance
except Exception:
    ROLE = "<arn:aws:iam::ACCOUNT:role/your-sagemaker-execution-role>"   # running elsewhere: set it explicitly

# ---- what to evaluate ----
BASE_HF = "Qwen/Qwen3.5-4B"
MODEL_PKG_ARN = "<model package ARN printed by step 03 of the training notebook>"
REFERENCE_BEDROCK_MODEL = "us.anthropic.claude-sonnet-4-6"   # optional reference row via Bedrock; None to skip
ENABLE_THINKING = False
EVAL_TASKS = ["product", "shop", "voucher"]   # held-out problems: 103 / 105 / 116
TRAIN_CORPUS = "oro-ai/sn15-shoppingbench-sft-15k"
MAX_PROBLEMS = None                           # e.g. 5 for a quick end-to-end test
RUN_IFEVAL = True

# ---- Endpoints (scripts/sm_endpoints.py) ----
REUSE_ENDPOINTS = {}          # e.g. {"base": "<endpoint name>"}: reuse InService endpoints, deploy the rest
SERVING = "vllm"              # "vllm": SageMaker vLLM container; "djl": DJL/LMI container
USE_MODELBUILDER = False      # deploy the student with the SDK's ModelBuilder instead (needs SERVING = "djl")
MAX_MODEL_LEN = 40960
# Below instances are tried in order of availability ; on InsufficientInstanceCapacity or CannotStartContainerError the next one is used.
INSTANCE_TYPES = {"djl": ["ml.g6e.xlarge", "ml.g6e.4xlarge", "ml.g5.2xlarge", "ml.p4d.24xlarge"],
                  "vllm": ["ml.g6e.xlarge", "ml.g6e.4xlarge", "ml.g6.2xlarge", "ml.g6e.2xlarge", "ml.p5.48xlarge"]}[SERVING]


# ---- Agent harness ----
ROLLOUT_THREADS = 12
WORK = Path.home() / "shopbench-eval"   # harness, index, venvs and outputs (~15 GB)
RUN_NAME = ""                           # use a new name per run: finished trajectories in OUT are reused
OUT = WORK / "out" / RUN_NAME if RUN_NAME else WORK / "out"
OUT.mkdir(parents=True, exist_ok=True)
RELAY_PORT = 8101                       # first local relay port

SCRIPTS = Path("scripts").resolve()
assert (SCRIPTS / "shopbench_harness.py").exists(), "run the notebook from its own folder (distillation/)"
sys.path.insert(0, str(SCRIPTS))
for name in ("botocore", "httpx", "httpx2"):
    logging.getLogger(name).setLevel(logging.WARNING)

def run(*args, **kw):
    """Run a command and stream its output into the notebook."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    for line in p.stdout: print(line, end="")
    if p.wait(): raise RuntimeError(f"exit code {p.returncode}: {args}")
print(REGION, ROLE)

## 1. Product search: Lucene index and search server

The tools query a **local copy of the catalog** through a search server, so every model sees the same results.

**The catalog.** `resources/documents.jsonl.gz` (1.4 GB) has one JSON record per product, 2,746,368 in total. 

A sample record in catalog:

```json
{"id": "126564",
 "contents": "Rainbow Loom Refill (Navy Blue Opaque)\nnavy blue 5 10\nloom bands 6 years and above ...",
 "product": {"product_id": "126564", "shop_id": "7889", "title": "Rainbow Loom Refill (Navy Blue Opaque)",
             "price": 75.0, "sold_count": 1, "service": ["official"],
             "sku_options": {"1": {"color family": "navy blue", ...}},
             "attributes": {"recommended_age": ["6 years and above"], ...}, ...}}
```

contents is the searchable text: the title, the SKU option values and the attribute values. 
product is the product's full details

Lucene Index: [Pyserini](https://github.com/castorini/pyserini), a Python toolkit over the
[Lucene](https://lucene.apache.org/) search library (Java, hence the JDK), builds an inverted index of `contents`. It
maps each word, lower-cased and stemmed, to the products that contain it. It also stores each raw record, so a product
can be fetched by its `id`. The index is 3.0 GB and needs to be built once.

Shopping Product Catalog Search Server : A Flask app that loads the index and listens on the port 
`127.0.0.1:5631`. 

Two routes, one per tool:

| Route (tool) | What it does |
|---|---|
| `/find_product` | 1. Ranks all products against `q` with **BM25**, the standard keyword-relevance score (rewards products containing the query's words, especially rarer ones, in shorter texts). <br>2. Walks down the ranking and drops products that fail the optional filters: `shop_id` (exact shop), `price` (`"low-high"`, e.g. `"100-500"`, `"1000-"`), `service` (each listed service must be offered). <br> 3. Keeps the **first 50** that pass.<br> 4. Optionally re-sorts those 50 by `sold_count` (`order`) or price (`priceasc` / `pricedesc`). <br> 5. Returns page `page` (1-5) of 10, each with `product_id`, `shop_id`, `title`, `price`, `service`, `sold_count` |
| `/view_product_information` | Looks up each ID in `product_ids` directly (no ranking). Returns `short_description`, `description`, `sku_options` and `attributes`; unknown IDs are skipped |

What this means for the agent:
- Search is **keyword** matching, not semantic. Wording the query like the product titles matters, and so does
  retrying with other words.
- Sorting only reorders the 50 most relevant matches. `priceasc` gives the cheapest *relevant* product, not the
  cheapest in the catalog.
- Size, color and other variant details are only visible after `view_product_information`. That is why the
  SKU/attribute checks in the scorer reward agents that inspect products before recommending.

If many tool calls fail (`null_tool_results` in the health checks), restart it with `harness.start_search()`.

In [ ]:
run("bash", SCRIPTS / "setup_shopbench.sh", WORK, env={**os.environ, "PYTHON": sys.executable})   # skips finished steps

from shopbench_harness import Harness, nlines
harness = Harness(WORK)
SB_DIR = harness.sb                     # the ShoppingBench checkout
if not harness.search_ok(): harness.start_search()
print("search server up")

## 2. Held-out test problems

147 of the 250 product test queries appear verbatim in the training corpus. With below script to make a clean split, only problems whose query never appears in TRAIN_CORPUS are evaluated
(`data/synthesize_<task>_test_clean.jsonl`).

In [ ]:
run(sys.executable, SCRIPTS / "make_clean_split.py", "--shopbench", SB_DIR, "--corpus", TRAIN_CORPUS, "--tasks", *EVAL_TASKS)
CLEAN = {t: SB_DIR / "data" / f"synthesize_{t}_test_clean.jsonl" for t in EVAL_TASKS}   # held-out problems
if MAX_PROBLEMS:                        # quick test: only the first N held-out problems of each intent
    for t in EVAL_TASKS:
        rows = open(CLEAN[t]).readlines()[:MAX_PROBLEMS]
        CLEAN[t] = CLEAN[t].with_name(f"synthesize_{t}_test_clean_first{MAX_PROBLEMS}.jsonl")
        CLEAN[t].write_text("".join(rows))
print({t: sum(1 for _ in open(p)) for t, p in CLEAN.items()})

## 3. Deploy the Base and Student Endpoints

- **Base**: The base model weights are managed by the deployment endpoint.
- **Student**: The post distillation merged weights of student model in the Model Package (checkpoints/hf_merged/).

**`ModelBuilder` alternative** (`USE_MODELBUILDER = True`, `SERVING = "djl"`): ModelBuilder resolves the container and
weights and manages the deployment endpoint.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from sm_endpoints import EndpointDeployer
DEP = EndpointDeployer(REGION, ROLE, serving=SERVING, instance_types=INSTANCE_TYPES, max_model_len=MAX_MODEL_LEN)

TO_DEPLOY = {"base": lambda: DEP.deploy("base", BASE_HF),
             "student": (lambda: DEP.deploy_student_with_modelbuilder(MODEL_PKG_ARN, INSTANCE_TYPES[-1]))
                        if USE_MODELBUILDER else
                        (lambda: DEP.deploy("student", "/opt/ml/model", DEP.student_model_data(MODEL_PKG_ARN)))}
ENDPOINTS = dict(REUSE_ENDPOINTS)

for ep in REUSE_ENDPOINTS.values():
    assert DEP.status(ep) == "InService", f"{ep} is not InService"
with ThreadPoolExecutor(2) as pool:
    futs = {tag: pool.submit(fn) for tag, fn in TO_DEPLOY.items() if tag not in ENDPOINTS}
    ENDPOINTS.update({tag: f.result() for tag, f in futs.items()})
print(ENDPOINTS)

### Connect the harness to the endpoints

The harness uses a plain OpenAI client; SageMaker endpoints need a short-lived bearer token. A small local relay per
model adds a fresh token to every request (or calls Bedrock for the reference). The smoke test checks the thinking
switch reaches base and student: `reasoned` should be `True` with thinking on and `False` with it off.

In [ ]:
from openai import OpenAI
import sm_openai_relay as relay
relay.stop_all()                                   # re-running this cell: free the ports first
URLS = {tag: relay.start_relay(tag, ep, RELAY_PORT + i, REGION) for i, (tag, ep) in enumerate(ENDPOINTS.items())}
if REFERENCE_BEDROCK_MODEL:
    URLS["teacher"] = relay.start_relay("teacher", REFERENCE_BEDROCK_MODEL, RELAY_PORT + 98, REGION, transport="bedrock")

from shopbench_harness import has_reasoning
for tag, url in URLS.items():
    c = OpenAI(base_url=url, api_key="unused")
    ask = lambda **kw: c.chat.completions.create(model="", max_tokens=1024, messages=[{"role": "user", "content": "What is 17*3?"}], **kw).choices[0]
    if tag == "teacher":                      # Bedrock: no thinking switch; just check the relay answers
        print(f"{tag:8s} connected: {ask().message.content[-40:]!r}")
        continue
    for flag in (True, False):
        ch = ask(extra_body={"chat_template_kwargs": {"enable_thinking": flag}})
        out = ch.message.content
        reasoned = has_reasoning(out) or (ch.finish_reason == "length" and "</think>" not in out)  # cut off mid-thought
        print(f"{tag:8s} enable_thinking={flag!s:5s} reasoned: {reasoned!s:5s}  {out[-40:]!r}")
# Expected: reasoned True with enable_thinking=True and False with False, for base and student.

## 4. Run the agents and score them (ASR)

All models run every held-out problem through the unmodified ShoppingBench agent loop, in parallel. Interrupting stops
the runs; re-running resumes (finished problems in `OUT` are kept, so use a new `RUN_NAME` for a new model).

In [ ]:
JOBS = {}
for tag, url in URLS.items():
    for t in EVAL_TASKS:
        cfg, rollout = harness.write_rollout_cfg(OUT, tag, t, CLEAN[t], url, ENABLE_THINKING, threads=ROLLOUT_THREADS)
        JOBS[f"{tag}_{t}"] = (cfg, rollout, nlines(CLEAN[t]))
harness.run_rollouts(JOBS)

In [ ]:
import pandas as pd
RESULTS = {}
for tag in URLS:
    for t in EVAL_TASKS:
        rollout = JOBS[f"{tag}_{t}"][1]
        RESULTS[(t, tag)] = {**harness.score(t, rollout, CLEAN[t]), **{f"health:{k}": v for k, v in harness.health(rollout, CLEAN[t]).items()}}
df = pd.DataFrame([{"task": t, "model": tag, **r} for (t, tag), r in RESULTS.items()]).set_index(["task", "model"]).sort_index()
for t in EVAL_TASKS:
    if (t, "base") in RESULTS and (t, "student") in RESULTS:
        b, s = RESULTS[(t, "base")]["asr"], RESULTS[(t, "student")]["asr"]
        line = f"{t}: ASR base {b:.3f} -> student {s:.3f} ({100 * (s - b):+.1f} pts)"
        if (t, "teacher") in RESULTS:
            te = RESULTS[(t, "teacher")]["asr"]
            line += f"; reference {te:.3f}" + (f"; student closes {(s - b) / (te - b):.0%} of the gap" if te > b else "")
        print(line)

# Avg.: mean over intents (the paper's "domain-weighted" average; ours excludes Knowledge and uses held-out problems)
avg = {tag: sum(RESULTS[(t, tag)]["asr"] for t in EVAL_TASKS) / len(EVAL_TASKS)
       for tag in URLS if all((t, tag) in RESULTS for t in EVAL_TASKS)}
if len(EVAL_TASKS) > 1:
    print(f"Avg. ASR ({', '.join(EVAL_TASKS)}; held-out): " + ", ".join(f"{tag} {a:.3f}" for tag, a in avg.items()))
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

LABELS = {"base": "base", "student": "student", "teacher": "reference (Bedrock)"}
models = [m for m in ("base", "student", "teacher") if any((t, m) in RESULTS for t in EVAL_TASKS)]
groups = EVAL_TASKS + (["Avg."] if len(EVAL_TASKS) > 1 else [])

def value(metric, t, m):
    if t == "Avg.":
        vals = [RESULTS[(x, m)][metric] for x in EVAL_TASKS if (x, m) in RESULTS]
        return sum(vals) / len(vals) if len(vals) == len(EVAL_TASKS) else np.nan
    return RESULTS[(t, m)][metric] if (t, m) in RESULTS else np.nan

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
panels = [("asr", "ASR (share of problems solved)"),
          ("recommend", "Share of trajectories that recommended a product")]
x, w = np.arange(len(groups)), 0.8 / len(models)
for ax, (metric, title) in zip(axes, panels):
    for i, m in enumerate(models):
        vals = [value(metric, t, m) for t in groups]
        bars = ax.bar(x + (i - (len(models) - 1) / 2) * w, vals, w, label=LABELS[m])
        ax.bar_label(bars, labels=["" if np.isnan(v) else f"{v:.2f}" for v in vals], fontsize=8, padding=2)
    ax.set_xticks(x, groups); ax.set_ylim(0, 1.05); ax.set_title(title); ax.grid(axis="y", alpha=0.3)
axes[0].legend(loc="upper right")
fig.suptitle("Held-out ShoppingBench problems" + (" (thinking on)" if ENABLE_THINKING else " (thinking off)"))
fig.tight_layout()
plt.show()

**Reading the results**

- Compare base and student on `asr` (`n` = problems scored); `exact_id` = exact gold product; `recommend` = recommended
  any real product.
- Health (share of trajectories): `empty_content` and `null_tool_results` should be ~0 (endpoint or search errors);
  `no_terminate` = ended without finishing, mostly an unparsable reply (distillation lowers it);
  `steps_with_reasoning` ≈ 1 with thinking on, ≈ 0 off.

Reference run (LoRA r64, held-out, thinking on):

| Model | product | shop | voucher | Avg. |
|---|---|---|---|---|
| base | 0.379 | 0.114 | 0.095 | 0.196 |
| student | 0.466 | 0.343 | 0.129 | 0.313 |

Shop gains most (`no_terminate` 0.80 → 0.37). Expect a few points of run-to-run variation.

## 5. IFEval (optional)

[IFEval](https://arxiv.org/abs/2311.07911) checks verifiable format instructions ;
`prompt_strict` = share of prompts with every rule followed. It runs with thinking **off** , and `reasoning_leaks` must be 0 for a valid score. A small drop vs. base is the expected cost of
specializing.

In [ ]:
if RUN_IFEVAL:
    LMEVAL_ENV = WORK / "lmevalenv"
    if not (LMEVAL_ENV / "bin/lm_eval").exists():
        run(sys.executable, "-m", "venv", LMEVAL_ENV)
        run(LMEVAL_ENV / "bin/pip", "install", "-q", "lm_eval[api,ifeval]")
    IFEVAL = {}
    for i, (tag, ep) in enumerate(ENDPOINTS.items()):
        url = relay.start_relay(f"ifeval_{tag}", ep, RELAY_PORT + 100 + i, REGION,
                                force={"chat_template_kwargs": {"enable_thinking": False}})
        out = subprocess.run([sys.executable, SCRIPTS / "run_ifeval.py", "--base-url", url, "--out", OUT / f"ifeval_{tag}",
                              "--lm-eval", LMEVAL_ENV / "bin/lm_eval"], capture_output=True, text=True)
        if out.returncode: raise RuntimeError(out.stderr[-3000:])
        IFEVAL[tag] = json.loads(out.stdout.strip().splitlines()[-1])
        print(tag, IFEVAL[tag], "" if IFEVAL[tag]["reasoning_leaks"] == 0 else "<- INVALID: reasoning leaked into predictions")

## 6. Delete the endpoints

Deletes every endpoint in `ENDPOINTS`, **including reused ones** (remove an entry to keep it), and stops the search
server and relays.

In [ ]:
for ep in ENDPOINTS.values():
    DEP.delete(ep, wait=False); print("deleted", ep)
harness.stop_all()
relay.stop_all()